# Frame Synchronization & Canonical Timeline

Notebook này tạo một bảng synchronized theo `frame_id` cho từng trip, gom về cùng một timeline:

- telemetry từ `ego`
- driver labels / confidence-like fields nếu có
- risk / TTC ground truth nếu có
- target summary
- modality availability theo frame
- event log gần nhất theo timestamp

Mục tiêu là tạo nền tảng cho API, analytics và unified event schema về sau.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "AGENTS.md").exists():
    if (PROJECT_ROOT.parent / "AGENTS.md").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
    else:
        raise FileNotFoundError("Cannot locate project root from the current notebook working directory.")

NOTEBOOK_HELPERS = PROJECT_ROOT / "notebooks"
if str(NOTEBOOK_HELPERS) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HELPERS))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import fleetiq_notebook_utils as nb

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

In [ ]:
practice_trip = "T01-Sample"
practice_dir = nb.practice_root() / practice_trip
sync_df = nb.build_sync_frame_table(practice_dir)
sync_df.head()

In [ ]:
nb.sync_quality_summary(sync_df)

In [ ]:
nb.build_event_log_table(practice_dir)

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)

axes[0].plot(sync_df["timestamp"], sync_df["speed_kmh"], color="#034EA2", linewidth=1.8)
axes[0].set_ylabel("speed_kmh")
axes[0].set_title(f"Synchronized timeline | {practice_trip}")

axes[1].plot(sync_df["timestamp"], sync_df["alertness_score"], color="#F37021", linewidth=1.8)
axes[1].set_ylabel("alertness")

axes[2].plot(sync_df["timestamp"], sync_df["min_ttc"], color="#D64545", linewidth=1.8)
axes[2].set_ylabel("min_ttc")

axes[3].plot(sync_df["timestamp"], sync_df["final_risk_score"], color="#19226D", linewidth=1.8)
axes[3].set_ylabel("risk")
axes[3].set_xlabel("timestamp (s)")

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.tight_layout()

In [ ]:
target_class_counts = nb.count_target_classes(sync_df)
target_class_counts

In [ ]:
frame_id = 300
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, stream in zip(axes, ["left", "right", "driver"]):
    image = nb.load_trip_image(practice_dir, frame_id, stream=stream)
    ax.imshow(image)
    ax.set_title(f"{stream} | frame {frame_id}")
    ax.axis("off")
plt.tight_layout()

In [ ]:
redacted_trip = "T01d"
redacted_dir = nb.redacted_root() / redacted_trip
redacted_sync = nb.build_sync_frame_table(redacted_dir)

redacted_sync[
    [
        "trip_id",
        "frame_id",
        "timestamp",
        "speed_kmh",
        "driver_state",
        "min_ttc",
        "final_risk_score",
        "target_count",
        "target_classes",
        "has_left_image",
        "has_driver_image",
    ]
].head()

In [ ]:
out_dir = PROJECT_ROOT / "artifacts" / "sync_tables"
out_dir.mkdir(parents=True, exist_ok=True)
sync_df.to_csv(out_dir / f"{practice_trip}_sync.csv", index=False)
redacted_sync.to_csv(out_dir / f"{redacted_trip}_sync.csv", index=False)
out_dir